 # 07 — Gaussian Smoothing Sensitivity Analysis



 This notebook evaluates whether Gaussian smoothing improves selected final

 functional classifiers from the thesis-ready model set.



 The notebook keeps the model set small and directly comparable to the final

 functional classification notebook.



 Compared representations:



 - raw L2-normalised sampled spectra;

 - Gaussian-smoothed spectra with selected sigma values, followed by

   L2 normalisation.



 Compared models:



 - weighted kNN with standardized Euclidean distance;

 - functional logistic regression with L2 regularisation;

 - functional linear SVM.



 Main outputs:



 - `smoothing_fold_metrics.csv`;

 - `smoothing_summary.csv`;

 - model coefficient storage for later interpretation.

In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import wilcoxon
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")



 ## 1. General configuration



 This section defines the main settings used throughout the smoothing

 experiment.



 - `RANDOM_STATE` makes repeated operations reproducible.

 - `SMOKE` can be set to `True` to run only a smaller test subset.

 - `SIGMAS` contains all tested Gaussian smoothing values.

 - `BEST_SIGMA` stores the selected smoothing value for later interpretation.

 - `DISTANCE_K` defines the number of neighbours for weighted kNN.

 - `INNER_CV` and `CS_GRID` define the inner hyperparameter search for the

   linear models.

In [ ]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

SMOKE = False

SIGMAS = [0.0, 4.0, 6.0, 8.0, 10.0, 12.0]
BEST_SIGMA = 8.0

DISTANCE_K = 5

INNER_CV = 3
CS_GRID = np.logspace(-2, 2, 6)

LINEAR_MODELS = [
    {
        "family": "Functional linear",
        "method": "Functional logistic regression (L2)",
        "kind": "logreg_l2",
    },
    {
        "family": "Functional linear",
        "method": "Functional linear SVM",
        "kind": "linear_svm",
    },
]



 ## 2. Paths and input files



 The notebook expects the input files to be stored in `og_data`.



 Required files:



 - `og_xp.csv`: labels and `source_id`;

 - `xp_sampled_spectra.csv`: sampled spectra;

 - `splits_rskf.json`: predefined repeated stratified CV splits.



 Outputs are saved to `results/07_smoothing`.

In [ ]:
BASE_DIR = Path.cwd() / "og_data"
OUT_DIR = Path.cwd() / "results" / "07_smoothing"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_CANDIDATES = [
    BASE_DIR / "og_xp.csv",
]

SPLIT_CANDIDATES = [
    BASE_DIR / "splits_rskf.json",
]

SAMPLED_CANDIDATES = [
    BASE_DIR / "xp_sampled_spectra.csv",
]



 ## 3. Plot style



 This section defines a consistent thesis-ready visual style. If figures are

 added later, they should be saved immediately after their plotting block using

 `show_and_save`.

In [ ]:
COLOR_RAW = "#104A7E"
COLOR_SMOOTH = "#78003F"
COLOR_DARK = "#0D1530"
COLOR_GREY = "#8C8C8C"
COLOR_POS = "#6193CD"
COLOR_NEG = "#6F0037"
COLOR_LIGHT = "#D9D9D9"

plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.linewidth": 0.8,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 9,
        "figure.dpi": 120,
    }
)



 ## 4. Helper functions



 These functions handle file lookup, fold ordering, smoothing, normalisation,

 metric calculation, result summarisation, plotting style, and figure export.

In [ ]:
def find_first_existing(paths: list[Path]) -> Path | None:
    """Return the first existing path from a list of candidate paths.

    Parameters
    ----------
    paths:
        Candidate file paths.

    Returns
    -------
    Path | None
        First existing file path, or `None` if no path exists.
    """
    for path in paths:
        if path.exists():
            return path

    return None


def split_sort_key(split_name: str) -> tuple[int, int]:
    """Sort repeated CV split names by repetition and fold number.

    The expected split name format is similar to `rep0_fold0`.

    Parameters
    ----------
    split_name:
        Name of one repeated stratified CV split.

    Returns
    -------
    tuple[int, int]
        Parsed repetition and fold numbers.
    """
    rep = int(split_name.split("_")[0].replace("rep", ""))
    fold = int(split_name.split("_")[1].replace("fold", ""))

    return rep, fold


def mean_std_string(mean: float, std: float) -> str | float:
    """Format a metric as `mean ± std`.

    Parameters
    ----------
    mean:
        Metric mean.
    std:
        Metric standard deviation.

    Returns
    -------
    str | float
        Formatted string, or `np.nan` when the mean is missing.
    """
    if pd.isna(mean):
        return np.nan

    std_value = 0.0 if pd.isna(std) else std
    return f"{mean:.4f} ± {std_value:.4f}"


def l2_normalise_rows(X: np.ndarray) -> np.ndarray:
    """Apply row-wise L2 normalisation.

    Each spectrum is divided by its L2 norm. Zero-norm rows are returned as
    zeros.

    Parameters
    ----------
    X:
        Matrix where rows are spectra and columns are wavelengths.

    Returns
    -------
    np.ndarray
        L2-normalised matrix.
    """
    row_norm = np.linalg.norm(X, axis=1, keepdims=True)

    return np.divide(
        X,
        row_norm,
        out=np.zeros_like(X),
        where=row_norm > 1e-20,
    )


def gaussian_smoothing_matrix(
    wavelengths: np.ndarray,
    sigma: float,
) -> np.ndarray:
    """Build a row-normalised Gaussian smoothing matrix.

    Each row contains Gaussian weights centred at one wavelength. The rows are
    normalised so that each smoothed wavelength is a weighted average of the
    original spectrum.

    Parameters
    ----------
    wavelengths:
        Wavelength grid.
    sigma:
        Gaussian kernel width.

    Returns
    -------
    np.ndarray
        Row-normalised smoothing matrix.
    """
    weights = np.exp(
        -0.5 * ((wavelengths[:, None] - wavelengths[None, :]) / sigma) ** 2
    )
    weights /= weights.sum(axis=1, keepdims=True)

    return weights


def build_representation(
    X_raw: np.ndarray,
    wavelengths: np.ndarray,
    sigma: float,
) -> np.ndarray:
    """Build raw or Gaussian-smoothed L2-normalised spectra.

    If `sigma` is zero, the original raw spectra are used. Otherwise, Gaussian
    smoothing is applied first and L2 normalisation is applied afterwards.

    Parameters
    ----------
    X_raw:
        Raw sampled spectra.
    wavelengths:
        Wavelength grid.
    sigma:
        Smoothing parameter. Zero means no smoothing.

    Returns
    -------
    np.ndarray
        Raw or smoothed L2-normalised representation.
    """
    if sigma == 0:
        X_work = X_raw.copy()
    else:
        smoothing_matrix = gaussian_smoothing_matrix(wavelengths, sigma)
        X_work = X_raw @ smoothing_matrix.T

    return l2_normalise_rows(X_work)


def normalize_scores_train_ref(
    scores_te: np.ndarray,
    scores_tr: np.ndarray,
) -> np.ndarray:
    """Normalise scores using the training-score range.

    Test scores are scaled using only the training minimum and maximum. This
    avoids leakage from the test fold during threshold selection.

    Parameters
    ----------
    scores_te:
        Scores to normalise.
    scores_tr:
        Training scores used as the reference range.

    Returns
    -------
    np.ndarray
        Scores clipped to the interval `[0, 1]`.
    """
    lo = float(np.min(scores_tr))
    hi = float(np.max(scores_tr))

    if hi == lo:
        return np.full_like(scores_te, 0.5, dtype=np.float64)

    normalised = (scores_te - lo) / (hi - lo)

    return np.clip(normalised, 0.0, 1.0).astype(np.float64)


def pick_youden_threshold(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    grid_size: int = 200,
) -> float:
    """Choose a classification threshold by maximising Youden's J.

    Youden's J is defined as sensitivity + specificity - 1. The threshold is
    selected on training-fold scores.

    Parameters
    ----------
    y_true:
        True binary labels.
    y_prob:
        Normalised class-1 scores.
    grid_size:
        Number of threshold candidates between 0 and 1.

    Returns
    -------
    float
        Selected threshold.
    """
    thresholds = np.linspace(0, 1, grid_size)
    best_j = -1.0
    best_threshold = 0.5

    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(np.int64)

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1],
        ).ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
        specificity = tn / (tn + fp) if (tn + fp) else 0.0
        youden_j = sensitivity + specificity - 1.0

        if youden_j > best_j:
            best_j = youden_j
            best_threshold = float(threshold)

    return best_threshold


def fold_metrics(
    y_true_te: np.ndarray,
    y_score_te: np.ndarray,
    y_true_tr: np.ndarray,
    y_score_tr: np.ndarray,
) -> dict[str, float]:
    """Calculate fold-level classification metrics.

    PR-AUC and ROC-AUC are calculated from continuous scores. Binary metrics
    are calculated after selecting a Youden threshold on the training fold.

    Parameters
    ----------
    y_true_te:
        True labels for the test fold.
    y_score_te:
        Continuous scores for the test fold.
    y_true_tr:
        True labels for the training fold.
    y_score_tr:
        Continuous scores for the training fold.

    Returns
    -------
    dict[str, float]
        Fold-level metric dictionary.
    """
    metrics = {
        "pr_auc": average_precision_score(y_true_te, y_score_te),
    }

    try:
        metrics["roc_auc"] = float(roc_auc_score(y_true_te, y_score_te))
    except ValueError:
        metrics["roc_auc"] = np.nan

    prob_tr = normalize_scores_train_ref(y_score_tr, y_score_tr)
    prob_te = normalize_scores_train_ref(y_score_te, y_score_tr)

    threshold = pick_youden_threshold(y_true_tr, prob_tr)
    y_pred = (prob_te >= threshold).astype(np.int64)

    metrics["youden_threshold"] = threshold
    metrics["sensitivity"] = recall_score(
        y_true_te,
        y_pred,
        pos_label=1,
        zero_division=0,
    )
    metrics["precision"] = precision_score(
        y_true_te,
        y_pred,
        pos_label=1,
        zero_division=0,
    )
    metrics["specificity"] = recall_score(
        y_true_te,
        y_pred,
        pos_label=0,
        zero_division=0,
    )
    metrics["accuracy"] = accuracy_score(y_true_te, y_pred)
    metrics["f1"] = f1_score(
        y_true_te,
        y_pred,
        pos_label=1,
        zero_division=0,
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true_te,
        y_pred,
        labels=[0, 1],
    ).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    metrics["youden_j"] = sensitivity + specificity - 1.0

    return metrics


def summarise_run(
    df_run: pd.DataFrame,
    group_cols: list[str],
) -> pd.DataFrame:
    """Summarise fold-level metrics by selected grouping columns.

    Parameters
    ----------
    df_run:
        Fold-level metric table.
    group_cols:
        Columns used for grouping.

    Returns
    -------
    pd.DataFrame
        Summary table with metric means and standard deviations.
    """
    metric_cols = [
        "pr_auc",
        "roc_auc",
        "sensitivity",
        "precision",
        "specificity",
        "accuracy",
        "f1",
        "youden_j",
        "youden_threshold",
    ]

    agg_dict = {}

    for metric in metric_cols:
        agg_dict[f"{metric}_mean"] = pd.NamedAgg(
            column=metric,
            aggfunc="mean",
        )
        agg_dict[f"{metric}_std"] = pd.NamedAgg(
            column=metric,
            aggfunc="std",
        )

    return df_run.groupby(group_cols).agg(**agg_dict).reset_index()


def apply_clean_axes(
    ax,
    add_grid: bool = False,
) -> None:
    """Apply minimal thesis-style formatting to a matplotlib axis.

    Parameters
    ----------
    ax:
        Matplotlib axis object.
    add_grid:
        Whether to add a light grid.
    """
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if add_grid:
        ax.grid(alpha=0.20, linewidth=0.7)
    else:
        ax.grid(False)


def show_and_save(fig, path: Path) -> None:
    """Show a figure and immediately save it as an SVG file.

    This function should be used right after each plotting block, so there is
    no separate figure-saving section at the end.

    Parameters
    ----------
    fig:
        Matplotlib figure object.
    path:
        Output SVG path.
    """
    plt.tight_layout()
    plt.show()
    fig.savefig(path, format="svg", bbox_inches="tight", facecolor="white")
    plt.close(fig)



 ## 5. Locate and load core files



 This section checks that all required files are available, then loads labels,

 spectra, and predefined repeated stratified CV splits.

In [ ]:
LABEL_FILE = find_first_existing(LABEL_CANDIDATES)
SPLIT_FILE = find_first_existing(SPLIT_CANDIDATES)
SAMPLED_FILE = find_first_existing(SAMPLED_CANDIDATES)

if LABEL_FILE is None:
    raise FileNotFoundError("Missing `og_xp.csv`.")

if SPLIT_FILE is None:
    raise FileNotFoundError("Missing `splits_rskf.json`.")

if SAMPLED_FILE is None:
    raise FileNotFoundError("Missing `xp_sampled_spectra.csv`.")

print("LABEL_FILE:", LABEL_FILE)
print("SPLIT_FILE:", SPLIT_FILE)
print("SAMPLED_FILE:", SAMPLED_FILE)

df_labels = pd.read_csv(LABEL_FILE)
df_spec = pd.read_csv(SAMPLED_FILE)

if "source_id" not in df_labels.columns or "y" not in df_labels.columns:
    raise ValueError("`og_xp.csv` must contain `source_id` and `y`.")

if "source_id" not in df_spec.columns:
    raise ValueError("`xp_sampled_spectra.csv` must contain `source_id`.")

wl_cols = [col for col in df_spec.columns if col.startswith("wl_")]

if len(wl_cols) == 0:
    raise ValueError("No sampled spectrum columns found. Expected `wl_*`.")

wavelengths = np.array(
    [float(col.split("_")[1]) for col in wl_cols],
    dtype=np.float64,
)

df_m = df_labels[["source_id", "y"]].merge(
    df_spec[["source_id"] + wl_cols],
    on="source_id",
    how="inner",
    validate="one_to_one",
)

X_raw = df_m[wl_cols].to_numpy(dtype=np.float64)
y = df_m["y"].to_numpy(dtype=np.int64)

with open(SPLIT_FILE, encoding="utf-8") as file:
    splits = json.load(file)

split_names = sorted(splits.keys(), key=split_sort_key)

if SMOKE:
    split_names = [
        split_name
        for split_name in split_names
        if split_name.startswith("rep0_")
    ]

print("Merged rows:", len(df_m))
print("X_raw shape:", X_raw.shape)
print("Binary fraction:", round((y == 1).mean(), 4))
print("Number of splits:", len(split_names))
print("Wavelength range:", wavelengths.min(), "to", wavelengths.max())



 ## 6. Build raw and smoothed representations



 Each representation is built from the original raw flux values:



 - `sigma = 0` means no smoothing;

 - `sigma > 0` means Gaussian smoothing using the selected sigma;

 - L2 normalisation is applied after smoothing.

In [ ]:
X_by_sigma = {}

for sigma in SIGMAS:
    X_by_sigma[sigma] = build_representation(X_raw, wavelengths, sigma)
    print(f"sigma={sigma}: shape={X_by_sigma[sigma].shape}")



 ## 7. Distance-based model helpers



 These functions implement weighted kNN with standardized Euclidean distance.

 The variance parameters are estimated from the training fold only.

In [ ]:
def train_seuclidean_params(X_tr: np.ndarray) -> dict[str, object]:
    """Estimate variance parameters for standardized Euclidean distance.

    Parameters
    ----------
    X_tr:
        Training feature matrix.

    Returns
    -------
    dict[str, object]
        Distance parameter dictionary containing feature-wise variances.
    """
    variances = np.var(X_tr, axis=0, ddof=1)
    variances = np.where(variances <= 1e-12, 1e-12, variances)

    return {
        "distance": "seuclidean",
        "V": variances,
    }


def pairwise_seuclidean(
    X_a: np.ndarray,
    X_b: np.ndarray,
    params: dict[str, object],
) -> np.ndarray:
    """Calculate standardized Euclidean pairwise distances.

    Parameters
    ----------
    X_a:
        First feature matrix.
    X_b:
        Second feature matrix.
    params:
        Parameter dictionary returned by `train_seuclidean_params`.

    Returns
    -------
    np.ndarray
        Pairwise distance matrix.
    """
    return pairwise_distances(
        X_a,
        X_b,
        metric="seuclidean",
        V=params["V"],
    )


def knn_scores_from_distances(
    distances: np.ndarray,
    y_ref: np.ndarray,
    k: int,
    weighted: bool = True,
    exclude_self: bool = False,
) -> np.ndarray:
    """Convert a distance matrix into kNN class-1 scores.

    For training-fold scores, `exclude_self=True` removes each object from its
    own neighbour set.

    Parameters
    ----------
    distances:
        Distance matrix between query and reference objects.
    y_ref:
        Labels of reference objects.
    k:
        Number of nearest neighbours.
    weighted:
        Whether to use inverse-distance weights.
    exclude_self:
        Whether to ignore diagonal self-distances.

    Returns
    -------
    np.ndarray
        Continuous class-1 scores.
    """
    distances = distances.copy()

    if exclude_self and distances.shape[0] == distances.shape[1]:
        np.fill_diagonal(distances, np.inf)

    neighbour_idx = np.argsort(distances, axis=1)[:, :k]
    neighbour_distances = np.take_along_axis(
        distances,
        neighbour_idx,
        axis=1,
    )
    neighbour_labels = y_ref[neighbour_idx]

    if weighted:
        weights = 1.0 / np.maximum(neighbour_distances, 1e-12)
        scores = (
            weights * neighbour_labels
        ).sum(axis=1) / weights.sum(axis=1)
    else:
        scores = neighbour_labels.mean(axis=1)

    return scores.astype(np.float64)


def run_weighted_knn_seuclidean_one_split(
    X: np.ndarray,
    y: np.ndarray,
    train_idx: np.ndarray,
    test_idx: np.ndarray,
) -> dict[str, float]:
    """Run weighted kNN with standardized Euclidean distance on one split.

    Parameters
    ----------
    X:
        Full representation matrix.
    y:
        Full binary label vector.
    train_idx:
        Training indices.
    test_idx:
        Test indices.

    Returns
    -------
    dict[str, float]
        Fold-level metric dictionary.
    """
    X_tr = X[train_idx]
    X_te = X[test_idx]
    y_tr = y[train_idx]
    y_te = y[test_idx]

    params = train_seuclidean_params(X_tr)

    distances_train = pairwise_seuclidean(X_tr, X_tr, params)
    distances_test = pairwise_seuclidean(X_te, X_tr, params)

    score_tr = knn_scores_from_distances(
        distances_train,
        y_tr,
        k=DISTANCE_K,
        weighted=True,
        exclude_self=True,
    )

    score_te = knn_scores_from_distances(
        distances_test,
        y_tr,
        k=DISTANCE_K,
        weighted=True,
        exclude_self=False,
    )

    return fold_metrics(y_te, score_te, y_tr, score_tr)



 ## 8. Functional linear model helpers



 These functions train functional logistic regression and functional linear

 SVM models directly on raw or smoothed spectra. Standardisation is fitted

 only on the training fold.

In [ ]:
def mean_inner_cv_roc_auc_at_best_c(
    clf: LogisticRegressionCV,
) -> float:
    """Return mean inner-CV ROC-AUC for the selected logistic-regression C.

    Parameters
    ----------
    clf:
        Fitted `LogisticRegressionCV` model.

    Returns
    -------
    float
        Mean inner-CV ROC-AUC at the selected value of `C`.
    """
    scores = clf.scores_

    if scores is None:
        return np.nan

    if isinstance(scores, dict):
        score_array = np.asarray(next(iter(scores.values())))
    elif hasattr(scores, "ndim") and scores.ndim == 3:
        score_array = scores[0]
    else:
        score_array = np.asarray(scores)

    c_grid = np.asarray(clf.Cs_)
    best_c = float(clf.C_[0])
    best_idx = int(np.argmin(np.abs(c_grid - best_c)))

    return float(np.mean(score_array[:, best_idx]))


def run_linear_model_one_split(
    X: np.ndarray,
    y: np.ndarray,
    train_idx: np.ndarray,
    test_idx: np.ndarray,
    kind: str,
) -> tuple[dict[str, float], np.ndarray]:
    """Run one functional linear model on one CV split.

    Parameters
    ----------
    X:
        Full representation matrix.
    y:
        Full binary label vector.
    train_idx:
        Training indices.
    test_idx:
        Test indices.
    kind:
        Model type: `logreg_l2` or `linear_svm`.

    Returns
    -------
    tuple[dict[str, float], np.ndarray]
        Fold-level metrics and fitted coefficient vector.
    """
    X_tr = X[train_idx]
    X_te = X[test_idx]
    y_tr = y[train_idx]
    y_te = y[test_idx]

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_te_scaled = scaler.transform(X_te)

    if kind == "logreg_l2":
        model = LogisticRegressionCV(
            Cs=CS_GRID,
            cv=StratifiedKFold(
                n_splits=INNER_CV,
                shuffle=True,
                random_state=RANDOM_STATE,
            ),
            penalty="l2",
            solver="lbfgs",
            class_weight="balanced",
            scoring="roc_auc",
            max_iter=2000,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )

        model.fit(X_tr_scaled, y_tr)

        score_tr = model.predict_proba(X_tr_scaled)[:, 1]
        score_te = model.predict_proba(X_te_scaled)[:, 1]
        coef = model.coef_[0].copy()

        extra = {
            "best_C": float(model.C_[0]),
            "best_cv_roc_auc": mean_inner_cv_roc_auc_at_best_c(model),
            "n_nonzero_coefs": int(X.shape[1]),
        }

    elif kind == "linear_svm":
        inner_cv = StratifiedKFold(
            n_splits=INNER_CV,
            shuffle=True,
            random_state=RANDOM_STATE,
        )

        c_scores = []

        # LinearSVC does not provide a LogisticRegressionCV-like wrapper, so C
        # is selected manually using inner CV and ROC-AUC.
        for c_value in CS_GRID:
            fold_scores = []

            for inner_train_idx, inner_valid_idx in inner_cv.split(
                X_tr_scaled,
                y_tr,
            ):
                clf = LinearSVC(
                    C=c_value,
                    class_weight="balanced",
                    max_iter=10000,
                    random_state=RANDOM_STATE,
                )

                clf.fit(
                    X_tr_scaled[inner_train_idx],
                    y_tr[inner_train_idx],
                )

                validation_score = clf.decision_function(
                    X_tr_scaled[inner_valid_idx],
                )

                try:
                    auc = roc_auc_score(
                        y_tr[inner_valid_idx],
                        validation_score,
                    )
                except ValueError:
                    auc = np.nan

                fold_scores.append(auc)

            c_scores.append(np.nanmean(fold_scores))

        best_idx = int(np.nanargmax(c_scores))
        best_c = float(CS_GRID[best_idx])

        model = LinearSVC(
            C=best_c,
            class_weight="balanced",
            max_iter=10000,
            random_state=RANDOM_STATE,
        )

        model.fit(X_tr_scaled, y_tr)

        score_tr = model.decision_function(X_tr_scaled)
        score_te = model.decision_function(X_te_scaled)
        coef = model.coef_[0].copy()

        extra = {
            "best_C": best_c,
            "best_cv_roc_auc": float(c_scores[best_idx]),
            "n_nonzero_coefs": int(np.sum(np.abs(coef) > 1e-12)),
        }

    else:
        raise ValueError(f"Unsupported kind: {kind}")

    metrics = fold_metrics(y_te, score_te, y_tr, score_tr)
    metrics.update(extra)

    return metrics, coef



 ## 9. Run selected models for raw and smoothed spectra



 Each selected model is evaluated for each smoothing setting on the same

 predefined repeated stratified CV splits.

In [ ]:
records = []
coef_store = {
    sigma: {model_cfg["method"]: [] for model_cfg in LINEAR_MODELS}
    for sigma in SIGMAS
}

for sigma in SIGMAS:
    X = X_by_sigma[sigma]

    representation = (
        "Raw L2-normalised sampled spectra"
        if sigma == 0
        else f"Gaussian-smoothed L2-normalised spectra, sigma={sigma:g}"
    )

    print("\n" + "=" * 80)
    print(f"Representation: {representation}")
    print("=" * 80)

    for split_name in split_names:
        print(f"Current fold: {split_name}")

        train_idx = np.array(splits[split_name]["train"], dtype=int)
        test_idx = np.array(splits[split_name]["test"], dtype=int)

        print("  -> weighted kNN + seuclidean")

        metrics = run_weighted_knn_seuclidean_one_split(
            X=X,
            y=y,
            train_idx=train_idx,
            test_idx=test_idx,
        )

        row = {
            "split": split_name,
            "sigma": sigma,
            "representation": representation,
            "family": "Distance-based functional",
            "method": "weighted kNN + seuclidean",
            "tuning": f"k={DISTANCE_K}",
        }
        row.update(metrics)
        records.append(row)

        for model_cfg in LINEAR_MODELS:
            print(f"  -> {model_cfg['method']}")

            metrics, coef = run_linear_model_one_split(
                X=X,
                y=y,
                train_idx=train_idx,
                test_idx=test_idx,
                kind=model_cfg["kind"],
            )

            row = {
                "split": split_name,
                "sigma": sigma,
                "representation": representation,
                "family": model_cfg["family"],
                "method": model_cfg["method"],
                "tuning": "inner-CV C",
            }
            row.update(metrics)

            records.append(row)
            coef_store[sigma][model_cfg["method"]].append(coef)

df_fold = pd.DataFrame(records)

print("\nFold metrics shape:", df_fold.shape)
display(df_fold.head())



 ## 10. Summarise smoothing results



 This section calculates mean and standard deviation of all metrics for each

 smoothing setting and method.

In [ ]:
df_summary = summarise_run(
    df_fold,
    ["sigma", "representation", "family", "method", "tuning"],
)

df_summary = df_summary.sort_values(
    ["method", "sigma"],
).reset_index(drop=True)

print("\n=== SMOOTHING SUMMARY ===")
display(
    df_summary[
        [
            "sigma",
            "method",
            "f1_mean",
            "pr_auc_mean",
            "roc_auc_mean",
            "sensitivity_mean",
            "specificity_mean",
        ]
    ]
)



 ## 11. Calculate deltas against raw spectra



 For each method, every smoothed representation is compared against the raw

 representation (`sigma = 0`).

In [ ]:
delta_rows = []

for method in df_summary["method"].unique():
    raw_row = df_summary[
        (df_summary["method"] == method)
        & (df_summary["sigma"] == 0.0)
    ]

    if raw_row.empty:
        continue

    raw_row = raw_row.iloc[0]

    for _, row in df_summary[df_summary["method"] == method].iterrows():
        delta_rows.append(
            {
                "method": method,
                "sigma": row["sigma"],
                "f1_delta_vs_raw": row["f1_mean"] - raw_row["f1_mean"],
                "pr_auc_delta_vs_raw": (
                    row["pr_auc_mean"] - raw_row["pr_auc_mean"]
                ),
                "roc_auc_delta_vs_raw": (
                    row["roc_auc_mean"] - raw_row["roc_auc_mean"]
                ),
                "sensitivity_delta_vs_raw": (
                    row["sensitivity_mean"] - raw_row["sensitivity_mean"]
                ),
                "specificity_delta_vs_raw": (
                    row["specificity_mean"] - raw_row["specificity_mean"]
                ),
            }
        )

df_delta = pd.DataFrame(delta_rows)

print("\n=== DELTA SIGMA MINUS RAW ===")
display(df_delta)



 ## 12. Paired Wilcoxon tests: smoothed vs raw



 For each method and each non-zero smoothing setting, this section tests

 whether fold-level F1 and PR-AUC differ from the raw representation.

In [ ]:
wilcoxon_rows = []

for method in df_fold["method"].unique():
    raw_fold = df_fold[
        (df_fold["method"] == method)
        & (df_fold["sigma"] == 0.0)
    ].sort_values("split")

    for sigma in [value for value in SIGMAS if value != 0.0]:
        smooth_fold = df_fold[
            (df_fold["method"] == method)
            & (df_fold["sigma"] == sigma)
        ].sort_values("split")

        common_splits = sorted(
            set(raw_fold["split"]).intersection(set(smooth_fold["split"]))
        )

        raw_common = raw_fold[
            raw_fold["split"].isin(common_splits)
        ].sort_values("split")
        smooth_common = smooth_fold[
            smooth_fold["split"].isin(common_splits)
        ].sort_values("split")

        for metric in ["f1", "pr_auc"]:
            raw_values = raw_common[metric].to_numpy(dtype=float)
            smooth_values = smooth_common[metric].to_numpy(dtype=float)

            try:
                stat, p_value = wilcoxon(
                    smooth_values,
                    raw_values,
                    zero_method="wilcox",
                    alternative="two-sided",
                )
            except ValueError:
                stat = np.nan
                p_value = np.nan

            wilcoxon_rows.append(
                {
                    "method": method,
                    "sigma": sigma,
                    "metric": metric,
                    "n_pairs": len(raw_values),
                    "raw_mean": np.mean(raw_values),
                    "smooth_mean": np.mean(smooth_values),
                    "mean_diff_smooth_minus_raw": np.mean(
                        smooth_values - raw_values
                    ),
                    "wilcoxon_stat": stat,
                    "p_value": p_value,
                    "significant_0_05": (
                        bool(p_value < 0.05)
                        if not pd.isna(p_value)
                        else False
                    ),
                }
            )

df_wilcoxon = pd.DataFrame(wilcoxon_rows)

print("\n=== WILCOXON: SMOOTHED VS RAW ===")
display(df_wilcoxon)



 ## 13. Save table outputs



 Figures should be saved immediately after plotting blocks. This section saves

 only CSV outputs.

In [ ]:
fold_path = OUT_DIR / "smoothing_fold_metrics.csv"
summary_path = OUT_DIR / "smoothing_summary.csv"
delta_path = OUT_DIR / "smoothing_delta_sigma_minus_raw.csv"
wilcoxon_path = OUT_DIR / "smoothing_wilcoxon.csv"

df_fold.to_csv(fold_path, index=False)
df_summary.to_csv(summary_path, index=False)
df_delta.to_csv(delta_path, index=False)
df_wilcoxon.to_csv(wilcoxon_path, index=False)

print("Saved:")
print("-", fold_path)
print("-", summary_path)
print("-", delta_path)
print("-", wilcoxon_path)



 ## 14. Final quick view



 This compact table shows the key smoothing results as mean ± standard

 deviation.

In [ ]:
pretty_summary = pd.DataFrame(
    {
        "Sigma": df_summary["sigma"],
        "Method": df_summary["method"],
        "PR-AUC": [
            mean_std_string(mean, std)
            for mean, std in zip(
                df_summary["pr_auc_mean"],
                df_summary["pr_auc_std"],
            )
        ],
        "ROC-AUC": [
            mean_std_string(mean, std)
            for mean, std in zip(
                df_summary["roc_auc_mean"],
                df_summary["roc_auc_std"],
            )
        ],
        "F1": [
            mean_std_string(mean, std)
            for mean, std in zip(
                df_summary["f1_mean"],
                df_summary["f1_std"],
            )
        ],
    }
)

display(pretty_summary)